# 02 — Sentiment Analysis

Phase 3 of the news-sentiment quant project. Goals:

1. Pull a fresh batch of cnyes financial headlines via the Phase 2 scraper.
2. Score each headline with the **self-hosted vLLM** model (default: `gemma-4-31b-it-nvfp4`) using a structured 5-class prompt that maps to a continuous score.
3. (Below in this notebook) cross-check against a **FinBERT** baseline.
4. Aggregate to a daily per-ticker sentiment frame and persist under `data/sentiment/`.

All LLM calls go through the OpenAI-compatible client, so swapping models means changing only `LLM_MODEL` in `.env`.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from textwrap import dedent

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_collection.news_scraper import fetch_all_feeds  # noqa: E402

load_dotenv(REPO_ROOT / ".env")

MODEL = os.environ["LLM_MODEL"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
client = OpenAI(base_url=BASE_URL, api_key=os.environ["OPENAI_API_KEY"])

print("endpoint:", BASE_URL)
print("model:   ", MODEL)
print("available:", [m.id for m in client.models.list().data])

endpoint: http://192.168.50.20:8004/v1
model:    gemma-4-31b-it-nvfp4


available: ['gemma-4-31b-it-nvfp4']


## 1. Pull a fresh news batch

In [2]:
news = fetch_all_feeds()
print("shape:", news.shape)
# Keep only articles that linked to at least one ticker — those are the ones
# that will eventually feed the per-stock sentiment series.
tagged = news[news["tickers"] != ""].copy()
print("tagged:", tagged.shape)
tagged.head(3)

shape: (142, 7)
tagged: (84, 7)


,article_id,source,title,summary,url,published_at,tickers
6,36ae71f8a03fb76978900f94507f9eeac497268b,cnyes_headline,英特爾進場！馬斯克AI晶片工廠計畫升級 挑戰台積電地位,英特爾 (INTC-US) 周二 (7 日) 宣布，將加入馬斯克主導的「Terafab」大型...,https://news.cnyes.com/news/id/6409728,2026-04-07 14:09:56+00:00,2330.TW
8,948d967f9eca7f0f71fa37740855a4f62974867f,cnyes_headline,外資3月狂賣8300億元卻不急著跑？金管會揭一原因 密切關注國際情勢,美伊戰爭自今年 2 月底開打，戰事延宕導致 3 月全球市場上沖下洗，外陸資單月賣超上市櫃股票...,https://news.cnyes.com/news/id/6409707,2026-04-07 13:32:31+00:00,8306.TW
15,87c606221c3dd2e285565d75d7b163e72e8b3d44,cnyes_headline,台矽谷新創基地告捷！國發會SV Hub領軍3 AI新創 成功攻占北美大廠供應鏈,為加速達成「人工智慧島」戰略目標，國發會正積極推動「AI 新十大建設」方案，致力於布建完整的...,https://news.cnyes.com/news/id/6409654,2026-04-07 12:21:05+00:00,2025.TW


## 2. Prompt design

Asking an LLM for a raw float in `[-1, 1]` is unreliable — distributions cluster on round numbers and the model will frequently refuse or hedge. We instead ask for **one of five labels**, then map deterministically to a continuous score:

| Label | Score |
|---|---|
| `very_negative` | −1.0 |
| `negative` | −0.5 |
| `neutral` | 0.0 |
| `positive` | +0.5 |
| `very_positive` | +1.0 |

We also force `temperature=0` and a tiny `max_tokens` budget so the response is stable and cheap. JSON output is requested via the prompt itself (vLLM supports `response_format={"type": "json_object"}` for many models, but we keep this notebook portable across vLLM build flavors and parse JSON manually with a fallback).

In [3]:
LABEL_TO_SCORE = {
    "very_negative": -1.0,
    "negative": -0.5,
    "neutral": 0.0,
    "positive": 0.5,
    "very_positive": 1.0,
}

SYSTEM_PROMPT = dedent("""\
    You are a financial news sentiment classifier for the Taiwan stock market.
    Given a single headline (and optional summary), judge its likely short-term
    impact on the mentioned company's stock price.

    Reply with a single JSON object and nothing else:
      {"label": <one of: very_negative, negative, neutral, positive, very_positive>,
       "confidence": <float between 0 and 1>}

    Rules:
    - Use "neutral" for macro/political headlines with no clear company-level read.
    - Earnings beats, capacity expansion, large orders, upgrades -> positive/very_positive.
    - Earnings misses, layoffs, regulatory probes, downgrades -> negative/very_negative.
    - Do NOT explain. Do NOT add markdown fences. Just the JSON object.
""")


def score_headline(title: str, summary: str = "") -> dict:
    user_msg = title if not summary else f"標題：{title}\n摘要：{summary[:300]}"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        max_tokens=40,
    )
    raw = resp.choices[0].message.content.strip()
    # Strip ```json fences if the model adds them despite instructions.
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        obj = json.loads(raw)
        label = obj.get("label", "neutral")
        conf = float(obj.get("confidence", 0.0))
    except Exception:
        label, conf = "neutral", 0.0
    return {
        "label": label,
        "score": LABEL_TO_SCORE.get(label, 0.0),
        "confidence": conf,
        "raw": raw,
    }


# Quick smoke test on three hand-picked headlines.
for h in [
    "台積電 Q1 EPS 創新高，資本支出再上修",
    "金管會調查某金控海外踩雷，傳重罰",
    "行政院長今日召開記者會說明政策方向",
]:
    print(h, "->", score_headline(h))

台積電 Q1 EPS 創新高，資本支出再上修 -> {'label': 'very_positive', 'score': 1.0, 'confidence': 0.98, 'raw': '{"label": "very_positive", "confidence": 0.98}'}


金管會調查某金控海外踩雷，傳重罰 -> {'label': 'very_negative', 'score': -1.0, 'confidence': 0.95, 'raw': '{"label": "very_negative", "confidence": 0.95}'}


行政院長今日召開記者會說明政策方向 -> {'label': 'neutral', 'score': 0.0, 'confidence': 0.95, 'raw': '{"label": "neutral", "confidence": 0.95}'}


## 3. Score the live batch

We score a small slice (first 30 tagged articles) to keep the notebook quick. The full batch is the job of `src/features/sentiment.py` once we extract this logic in 3c.

In [4]:
sample = tagged.head(30).reset_index(drop=True).copy()
# Iterate explicitly to keep dict-of-records → DataFrame alignment trivial.
score_records = [
    score_headline(r["title"], r["summary"]) for _, r in sample.iterrows()
]
scores = pd.DataFrame(score_records)
scored = pd.concat([sample, scores], axis=1)
scored[["published_at", "tickers", "label", "score", "confidence", "title"]].head(15)

,published_at,tickers,label,score,confidence,title
0,2026-04-07 14:09:56+00:00,2330.TW,negative,-0.5,0.85,英特爾進場！馬斯克AI晶片工廠計畫升級 挑戰台積電地位
1,2026-04-07 13:32:31+00:00,8306.TW,neutral,0.0,0.90,外資3月狂賣8300億元卻不急著跑？金管會揭一原因 密切關注國際情勢
2,2026-04-07 12:21:05+00:00,2025.TW,positive,0.5,0.85,台矽谷新創基地告捷！國發會SV Hub領軍3 AI新創 成功攻占北美大廠供應鏈
3,2026-04-07 12:18:34+00:00,2025.TW,negative,-0.5,0.85,菲國稅改回溯鉅額補稅 金管會：台資銀行集體斡旋中 暫未繳納
4,2026-04-07 12:11:12+00:00,6285.TW,very_positive,1.0,0.95,鉅亨速報 - Factset 最新調查：啟碁(6285-TW)EPS預估上修至8.56元，預...
5,2026-04-07 12:10:21+00:00,4958.TW,positive,0.5,0.95,鉅亨速報 - Factset 最新調查：臻鼎-KY(4958-TW)目標價調升至245元，幅...
6,2026-04-07 11:54:54+00:00,"2026.TW,4958.TW",positive,0.5,0.95,臻鼎-KY首季營收407.28億元改寫歷史同期新高 年增1.61%
7,2026-04-07 11:44:07+00:00,"2026.TW,3371.TW,7820.TW,9795.TW",very_positive,1.0,0.98,立盈環保首季營收9795萬元改寫同期新高 年增63.3%
8,2026-04-07 11:41:00+00:00,2882.TW,positive,0.5,0.90,金融業唯一！國泰金與OpenAI啟動策略合作 打造企業級AI工作站
9,2026-04-07 11:39:09+00:00,2026.TW,very_positive,1.0,0.98,鮮活果汁-KY創史上最強首季營收達11.41億元 年增55.13%


In [5]:
# Label distribution sanity check.
scored["label"].value_counts()

label
very_positive    14
positive         11
neutral           3
negative          2
Name: count, dtype: int64

## 4. Per-ticker daily aggregation (preview)

Each article can mention multiple tickers, so we explode and group. The full pipeline (with confidence weighting and time-decay) lands in `src/features/sentiment.py` later in this phase.

In [6]:
exploded = (
    scored.assign(ticker=scored["tickers"].str.split(","))
          .explode("ticker")
          .assign(date=lambda d: pd.to_datetime(d["published_at"]).dt.tz_convert("Asia/Taipei").dt.date)
)

daily = (
    exploded.groupby(["date", "ticker"])
            .agg(mean_score=("score", "mean"),
                 mean_conf=("confidence", "mean"),
                 n_articles=("score", "size"))
            .reset_index()
            .sort_values(["date", "ticker"])
)
daily.head(15)

,date,ticker,mean_score,mean_conf,n_articles
0,2026-04-07,1342.TW,1.000000,0.980000,1
1,2026-04-07,2023.TW,0.000000,0.900000,1
2,2026-04-07,2024.TW,1.000000,0.950000,1
3,2026-04-07,2025.TW,0.000000,0.850000,2
4,2026-04-07,2026.TW,0.642857,0.948571,7
5,2026-04-07,2330.TW,-0.500000,0.850000,1
6,2026-04-07,2363.TW,0.500000,0.950000,1
7,2026-04-07,2882.TW,0.500000,0.900000,1
8,2026-04-07,2887.TW,0.500000,0.875000,2
9,2026-04-07,3167.TW,1.000000,0.980000,1


## Next steps

- Add FinBERT baseline cell below this section (Phase 3b).
- Extract the prompt + scoring loop into `src/features/sentiment.py` with batching, retry, and a CLI (Phase 3c).
- Persist the daily frame to `data/sentiment/`.